# Métricas y Monitoring para Visualizaciones

## Introducción

Este notebook explora el sistema de **métricas y monitoring** del módulo de visualización, que permite:

- **Logs detallados** de tiempo de exportación
- **Métricas de uso de memoria**
- **Alertas** para grafos muy grandes
- **Reportes** automáticos

---

## 1. Configuración Inicial

In [ ]:
import sys
sys.path.insert(0, '../..')

from pathlib import Path
from datetime import datetime
import asyncio
import random
import json

# Imports del módulo de métricas
from app.core.visualization.metrics_collector import (
    MetricsCollector,
    MetricsContext,
    ExportMetrics,
    BatchMetrics,
    Alert,
    AlertThresholds,
    AlertSeverity,
    MetricType,
    track_export
)

# Otros módulos relacionados
from app.core.visualization.optimizer import (
    VisualizationOptimizer,
    OptimizationConfig,
    OptimizationLevel
)

print("Módulos importados correctamente")

---

## 2. Conceptos Básicos del MetricsCollector

El `MetricsCollector` es el componente central que recolecta métricas durante las operaciones de exportación.

### 2.1 Crear un Colector

In [ ]:
# Crear colector con configuración por defecto
collector = MetricsCollector()

# Ver umbrales por defecto
thresholds = collector.thresholds
print(" Umbrales por defecto:")
print(f"   Tiempo - Warning: {thresholds.export_time_warning_ms}ms, Critical: {thresholds.export_time_critical_ms}ms")
print(f"   Memoria - Warning: {thresholds.memory_warning_mb}MB, Critical: {thresholds.memory_critical_mb}MB")
print(f"   Nodos - Warning: {thresholds.node_count_warning}, Critical: {thresholds.node_count_critical}")
print(f"   Aristas - Warning: {thresholds.edge_count_warning}, Critical: {thresholds.edge_count_critical}")

### 2.2 Umbrales Personalizados

In [ ]:
# Configurar umbrales personalizados
custom_thresholds = AlertThresholds(
    # Tiempos de exportación
    export_time_warning_ms=1000.0,    # 1 segundo
    export_time_critical_ms=5000.0,   # 5 segundos
    
    # Uso de memoria
    memory_warning_mb=128.0,          # 128 MB
    memory_critical_mb=256.0,         # 256 MB
    
    # Tamaño del grafo
    node_count_warning=100,           # 100 nodos
    node_count_critical=500,          # 500 nodos
    edge_count_warning=500,           # 500 aristas
    edge_count_critical=2000,         # 2000 aristas
    
    # Tamaño de archivo
    file_size_warning_mb=5.0,
    file_size_critical_mb=20.0,
    
    # Tasa de fallos
    failure_rate_warning=10.0,        # 10%
    failure_rate_critical=25.0        # 25%
)

collector_custom = MetricsCollector(thresholds=custom_thresholds)
print("Colector con umbrales personalizados creado")

---

## 3. Tracking de Exportaciones

### 3.1 Tracking Manual

In [ ]:
# Iniciar un batch de exportaciones
collector.start_batch("notebook-batch-001", total_tasks=3)

# Primera exportación
print("Exportación 1: Árbol de Recursión")
metrics1 = collector.start_export(
    task_id="export-fib",
    visualization_type="recursion_tree",
    export_format="svg",
    node_count=31,
    edge_count=30
)

# Simular trabajo
import time
time.sleep(0.1)

# Finalizar con éxito
collector.end_export(metrics1, success=True)
print(f"   Tiempo: {metrics1.processing_time_ms:.2f}ms")

# Segunda exportación
print("\n Exportación 2: Grafo con Optimización")
metrics2 = collector.start_export(
    task_id="export-graph",
    visualization_type="graph",
    export_format="png",
    node_count=100,
    edge_count=200
)

# Registrar optimización
collector.track_optimization(metrics2, optimization_time_ms=45.5)
time.sleep(0.08)

collector.end_export(metrics2, success=True)
print(f"   Tiempo: {metrics2.processing_time_ms:.2f}ms")
print(f"   Optimización: {metrics2.optimization_time_ms:.2f}ms")

# Tercera exportación (con error)
print("\n Exportación 3: Con Error")
metrics3 = collector.start_export(
    task_id="export-flow",
    visualization_type="execution_flow",
    export_format="dot"
)

time.sleep(0.05)

collector.end_export(metrics3, success=False, error="Error de renderizado")
print(f"   Error: {metrics3.error}")

# Finalizar batch
batch = collector.end_batch()
print("\n Batch completado:")
print(f"   Exitosas: {batch.successful_tasks}/{batch.total_tasks}")
print(f"   Throughput: {batch.throughput_tasks_per_sec:.2f} tareas/seg")

### 3.2 Tracking con Context Manager

El context manager `MetricsContext` simplifica el tracking automático:

In [ ]:
collector2 = MetricsCollector()
collector2.start_batch("context-batch", total_tasks=2)

# Exportación exitosa
with MetricsContext(
    collector2,
    task_id="ctx-export-1",
    visualization_type="graph",
    export_format="svg",
    node_count=50,
    edge_count=80
) as metrics:
    # El tracking inicia automáticamente
    time.sleep(0.1)
    # Al salir del with, se registra éxito automáticamente

print(f" Export 1: {metrics.processing_time_ms:.2f}ms")

# Exportación con error (capturado automáticamente)
try:
    with MetricsContext(
        collector2,
        task_id="ctx-export-2",
        visualization_type="tree",
        export_format="png"
    ) as metrics:
        time.sleep(0.05)
        raise ValueError("Error simulado")
except ValueError:
    pass

print(f" Export 2: Error capturado - {metrics.error}")

batch2 = collector2.end_batch()
print(f"\n Resultado: {batch2.successful_tasks} éxitos, {batch2.failed_tasks} fallos")

---

## 4. Sistema de Alertas

### 4.1 Configurar Alertas con Callback

In [ ]:
# Lista para capturar alertas
captured_alerts = []

def alert_handler(alert):
    """Callback que se ejecuta cuando se genera una alerta"""
    captured_alerts.append(alert)
    
    icons = {
        AlertSeverity.INFO: "INFO",
        AlertSeverity.WARNING: "WARNING",
        AlertSeverity.CRITICAL: "CRITICAL"
    }
    
    print(f"{icons[alert.severity]} Alerta: {alert.message}")

# Crear colector con umbrales bajos y callback
low_thresholds = AlertThresholds(
    node_count_warning=50,
    node_count_critical=100,
    edge_count_warning=100,
    edge_count_critical=200
)

alert_collector = MetricsCollector(
    thresholds=low_thresholds,
    alert_callback=alert_handler
)

print("Sistema de alertas configurado\n")

# Ejecutar exportaciones que dispararán alertas
alert_collector.start_batch("alert-demo", total_tasks=3)

print("1️Grafo pequeño (20 nodos):")
m1 = alert_collector.start_export("small", "graph", "svg", node_count=20, edge_count=30)
alert_collector.end_export(m1)
print("   (Sin alertas)")

print("\n2️Grafo mediano (75 nodos):")
m2 = alert_collector.start_export("medium", "graph", "svg", node_count=75, edge_count=120)
alert_collector.end_export(m2)

print("\n3️Grafo grande (150 nodos, 250 aristas):")
m3 = alert_collector.start_export("large", "graph", "svg", node_count=150, edge_count=250)
alert_collector.end_export(m3)

alert_collector.end_batch()

print(f"\n Total alertas capturadas: {len(captured_alerts)}")

### 4.2 Consultar y Resolver Alertas

In [ ]:
# Obtener todas las alertas
all_alerts = alert_collector.get_alerts()
print(f" Total alertas: {len(all_alerts)}")

# Filtrar por severidad
warnings = alert_collector.get_alerts(severity=AlertSeverity.WARNING)
criticals = alert_collector.get_alerts(severity=AlertSeverity.CRITICAL)

print(f"   WARNING Warnings: {len(warnings)}")
print(f"   CRITICAL Críticas: {len(criticals)}")

# Ver detalles de una alerta
if all_alerts:
    alert = all_alerts[0]
    print(f"\n Detalle de alerta:")
    print(f"   ID: {alert.id}")
    print(f"   Severidad: {alert.severity.value}")
    print(f"   Mensaje: {alert.message}")
    print(f"   Valor: {alert.value}")
    print(f"   Umbral: {alert.threshold}")
    print(f"   Resuelta: {alert.resolved}")
    
    # Resolver la alerta
    alert_collector.resolve_alert(alert.id)
    print(f"\n    Alerta {alert.id} resuelta")

# Ver alertas no resueltas
unresolved = alert_collector.get_alerts(unresolved_only=True)
print(f"\n Alertas sin resolver: {len(unresolved)}")

---

## 5. Métricas de Memoria

El sistema trackea el uso de memoria durante las exportaciones:

In [ ]:
memory_collector = MetricsCollector()
memory_collector.start_batch("memory-demo", total_tasks=3)

# Simular diferentes cargas de memoria
datasets = [
    ("pequeño", 100),
    ("mediano", 1000),
    ("grande", 5000)
]

for name, size in datasets:
    metrics = memory_collector.start_export(
        task_id=f"mem-{name}",
        visualization_type="graph",
        export_format="json",
        node_count=size
    )
    
    # Crear datos en memoria
    data = [{"id": i, "value": "x" * 100} for i in range(size)]
    time.sleep(0.05)
    
    memory_collector.end_export(metrics)
    del data
    
    mem_kb = metrics.memory_delta_bytes / 1024
    print(f" {name.capitalize()}: {size} items -> {mem_kb:.2f} KB de memoria")

batch = memory_collector.end_batch()

print(f"\n Memoria total usada: {batch.total_memory_used_bytes / 1024:.2f} KB")
print(f" Pico de memoria: {batch.peak_memory_bytes / (1024*1024):.2f} MB")

---

## 6. Estadísticas y Reportes

### 6.1 Obtener Estadísticas

In [ ]:
# Crear colector con datos de ejemplo
stats_collector = MetricsCollector()

# Ejecutar varios batches
for batch_num in range(3):
    stats_collector.start_batch(f"stats-batch-{batch_num}", total_tasks=5)
    
    for i in range(5):
        m = stats_collector.start_export(
            f"task-{batch_num}-{i}",
            "graph",
            "svg",
            node_count=random.randint(10, 100)
        )
        time.sleep(random.uniform(0.01, 0.05))
        stats_collector.end_export(m, success=random.random() > 0.1)
    
    stats_collector.end_batch()

# Obtener estadísticas
stats = stats_collector.get_statistics()

print("ESTADÍSTICAS GLOBALES")
print(f"Total exportaciones: {stats['total_exports']}")
print(f"Total batches: {stats['total_batches']}")
print()
print("Tiempos de Procesamiento:")
print(f"   Promedio: {stats['processing_time']['avg_ms']:.2f}ms")
print(f"   Mínimo: {stats['processing_time']['min_ms']:.2f}ms")
print(f"   Máximo: {stats['processing_time']['max_ms']:.2f}ms")
print(f"   Total: {stats['processing_time']['total_ms']:.2f}ms")
print()
print("Uso de Memoria:")
print(f"   Promedio: {stats['memory_usage']['avg_bytes'] / 1024:.2f} KB")
print(f"   Máximo: {stats['memory_usage']['max_bytes'] / 1024:.2f} KB")
print()
print("Alertas:")
print(f"   Total: {stats['alerts']['total']}")
print(f"   Sin resolver: {stats['alerts']['unresolved']}")

### 6.2 Historial de Batches

In [ ]:
history = stats_collector.get_batch_history()

print("HISTORIAL DE BATCHES")
print(f"{'Batch ID':<20} {'Tareas':<10} {'Éxito':<10} {'Throughput':<12}")

for batch in history:
    success_rate = batch.successful_tasks / batch.total_tasks * 100
    print(
        f"{batch.batch_id:<20} "
        f"{batch.total_tasks:<10} "
        f"{success_rate:>6.1f}%    "
        f"{batch.throughput_tasks_per_sec:>8.2f}/s"
    )

### 6.3 Exportar Reporte JSON

In [ ]:
# Crear directorio de reportes
reports_dir = Path("../../data/exports/reports")
reports_dir.mkdir(parents=True, exist_ok=True)

# Exportar reporte completo
report_path = reports_dir / f"metrics_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
stats_collector.export_report(report_path, include_raw_metrics=True)

print(f"Reporte guardado en: {report_path}")

# Ver estructura del reporte
with open(report_path) as f:
    report = json.load(f)
    
print("\nEstructura del reporte:")
print(f"   - generated_at: {report['generated_at']}")
print(f"   - statistics: {list(report['statistics'].keys())}")
print(f"   - alerts: {len(report['alerts'])} items")
print(f"   - batches: {len(report['batches'])} items")

---

## 7. Integración con Optimizador

El MetricsCollector puede trabajar junto con el `VisualizationOptimizer`:

In [ ]:
# Crear colector y optimizador
int_collector = MetricsCollector()
optimizer = VisualizationOptimizer(
    OptimizationConfig(level=OptimizationLevel.MODERATE)
)

int_collector.start_batch("integration-batch", total_tasks=3)

# Datos de grafos de prueba
test_graphs = [
    ("Pequeño", 20, 30),
    ("Mediano", 100, 180),
    ("Grande", 300, 600)
]

print("Optimización con Tracking de Métricas\n")

for name, num_nodes, num_edges in test_graphs:
    # Crear grafo
    nodes = [{"id": f"n{i}", "label": f"Node {i}"} for i in range(num_nodes)]
    edges = [(f"n{i}", f"n{(i+1) % num_nodes}", {}) for i in range(num_edges)]
    
    # Iniciar tracking
    metrics = int_collector.start_export(
        task_id=f"opt-{name.lower()}",
        visualization_type="graph",
        export_format="svg",
        node_count=num_nodes,
        edge_count=num_edges
    )
    
    # Aplicar optimización
    opt_start = time.time()
    opt_nodes, opt_edges, metadata = optimizer.optimize_graph(nodes, edges)
    opt_time = (time.time() - opt_start) * 1000
    
    # Registrar tiempo de optimización
    int_collector.track_optimization(metrics, opt_time)
    
    # Simular renderizado
    time.sleep(0.05)
    
    # Finalizar
    int_collector.end_export(metrics, success=True)
    
    # Mostrar resultados
    reduction_nodes = metadata['reduction_ratio']['nodes'] * 100
    reduction_edges = metadata['reduction_ratio']['edges'] * 100
    
    print(f" {name}:")
    print(f"   Nodos: {num_nodes} → {len(opt_nodes)} (-{reduction_nodes:.1f}%)")
    print(f"   Aristas: {num_edges} → {len(opt_edges)} (-{reduction_edges:.1f}%)")
    print(f"   Tiempo opt: {opt_time:.2f}ms")
    print()

batch = int_collector.end_batch()

print("Resumen:")
print(f"   Tiempo total: {batch.total_processing_time_ms:.2f}ms")
print(f"   Throughput: {batch.throughput_tasks_per_sec:.2f} tareas/seg")

---

## 8. Tipos de Métricas

El sistema define varios tipos de métricas a través de `MetricType`:

In [ ]:
print("TIPOS DE MÉTRICAS DISPONIBLES")

for metric_type in MetricType:
    descriptions = {
        MetricType.EXPORT_TIME: "Tiempo de exportación",
        MetricType.MEMORY_USAGE: "Uso de memoria",
        MetricType.GRAPH_SIZE: "Tamaño del grafo",
        MetricType.FILE_SIZE: "Tamaño de archivo",
        MetricType.OPTIMIZATION_TIME: "Tiempo de optimización",
        MetricType.RENDER_TIME: "Tiempo de renderizado",
        MetricType.BATCH_THROUGHPUT: "Throughput del batch"
    }
    
    print(f"   {metric_type.value}: {descriptions.get(metric_type, 'N/A')}")

---

## 9. Severidades de Alertas

In [ ]:
print("SEVERIDADES DE ALERTAS")

severity_info = {
    AlertSeverity.INFO: ("INFO", "Información general, sin acción requerida"),
    AlertSeverity.WARNING: ("WARNING", "Situación que requiere atención"),
    AlertSeverity.CRITICAL: ("CRITICAL", "Problema grave que requiere acción inmediata")
}

for severity, (icon, description) in severity_info.items():
    print(f"   {icon} {severity.value}: {description}")

---

## 10. Limpieza del Historial

In [ ]:
# Ver estado actual
print(f" Antes de limpiar:")
print(f"   Batches: {len(stats_collector.get_batch_history())}")
print(f"   Alertas: {len(stats_collector.get_alerts())}")

# Limpiar historial
stats_collector.clear_history()

print(f"\nDespués de limpiar:")
print(f"   Batches: {len(stats_collector.get_batch_history())}")
print(f"   Alertas: {len(stats_collector.get_alerts())}")

---

## 11. Mejores Prácticas

### Configuración de Umbrales

- **Tiempo**: Ajustar según la complejidad típica de tus visualizaciones
- **Memoria**: Considerar el entorno de ejecución (local vs servidor)
- **Tamaño de grafo**: Basarse en pruebas de rendimiento reales

### Uso del Context Manager

In [ ]:
# ✅ Recomendado: Usar MetricsContext para tracking automático
with MetricsContext(collector, task_id="...", ...) as metrics:
    # Tu código aquí
    pass

# ❌ Evitar: Olvidar llamar a end_export()
metrics = collector.start_export(...)
# ... código ...
# ¡No olvides collector.end_export(metrics)!

### Manejo de Alertas

In [ ]:
# Configurar callback para notificaciones en tiempo real
def notify_critical(alert):
    if alert.severity == AlertSeverity.CRITICAL:
        # Enviar email, log a sistema externo, etc.
        pass

collector = MetricsCollector(alert_callback=notify_critical)

### Reportes Periódicos

In [ ]:
# Generar reportes automáticamente después de cada batch
def save_batch_report(collector, batch_id):
    report_path = reports_dir / f"batch_{batch_id}.json"
    collector.export_report(report_path)

---

## Resumen

El sistema de **Métricas y Monitoring** proporciona:

| Característica | Descripción |
|---|---|
| **Tracking** | Métricas detalladas de cada exportación |
| **Memoria** | Monitoreo de uso de memoria |
| **Alertas** | Sistema configurable de alertas |
| **Reportes** | Exportación a JSON |
| **Integración** | Compatible con BatchExporter y Optimizer |
| **Context Manager** | Tracking automático con `MetricsContext` |

In [ ]:
print("Notebook completado")